Немного полезных функций

Загрузка данных из файла. Тут две полезные конструкции. 
Первая, это построчное чтение реализованное с помощью
~~~
for line in open(path+'/movies.dat'):
~~~
Вторая -- функция split(), позволяющая разделить строку, по символу или последовательности символов в качестве разделителя.


In [ ]:
#Чтение данных

def loadMovieLens1M(path='./ml-1m'):
# Получить названия фильмов
    movies={}
    for line in open(path+'/movies.dat', encoding="cp1251"):
        (id,title)=line.split('::')[0:2] #при желании тут можно добавить чтение жанров.
        movies[id]=title
# Загрузить данные
    prefs={}
    for line in open(path+'/ratings.dat', encoding="cp1251"):
        (user,movieid,rating,ts)=line.split('::')
        prefs.setdefault(user,{})
        prefs[user][movies[movieid]]=float(rating)
    return prefs

In [ ]:
def loadMovieLens(path='./ml-latest-small'):
# Получить названия фильмов
    movies={}
    skip_first = True
    for line in open(path+'/movies.csv'):
        if skip_first:
            skip_first = False
            continue
        (id,title)=line.split(',')[0:2]
        movies[id]=title
# Загрузить данные
    prefs={}
    skip_first = True
    for line in open(path+'/ratings.csv'):
        if skip_first:
            skip_first = False
            continue
        (user,movieid,rating,ts)=line.split(',')
        prefs.setdefault(user,{})
        prefs[user][movies[movieid]]=float(rating)
    return prefs

In [ ]:
%%time
critics = loadMovieLens1M()

In [ ]:
%%time
critics = loadMovieLens()

In [ ]:
len(critics)

Тестовая и обучающая выборки.
Рассмотрим выбор или не выбор элемента с заданной вероятностью, для этого нам понадобится модуль random и функция random(). Последняя возвращает случайное число равномерно распределенное в диапазоне от 0 до 1. Этим удобно пользоваться, чтобы получить случайное событие с вероятностью \alpha достаточно использовать конструкцию 
~~~
if random()<alpha:
~~~

Функция seed() инициализирует генератор псевдослучайных чисел заданным числом. Это позволяет воспроизводить идентичную последовательность случайных значений при последовательных запусков. Очень полезно при отладке.

In [ ]:
from random import random, seed
seed(100)
print(random(), random(), random())
print(random(), random(), random())
seed(100)
print(random(), random(), random())


In [ ]:
print(random(), random(), random())

Отберем, например, около 2 процентов пользователей:

In [ ]:
seed(20)
user_list = [u for u in critics if random()<1.02]

In [ ]:
len(user_list)

In [ ]:
sub_movies = {}
for u in user_list:
    for item in critics[u]:
        if not item in sub_movies:
            sub_movies[item] = 0
        sub_movies[item] += 1
            

In [ ]:
sub_movies

Оставим только фильмы которые оценило хотя бы 5 человек

In [ ]:
movie5 = [m for m in sub_movies if sub_movies[m]>4]

In [ ]:
len(movie5)

In [ ]:
movie5_sublist = [m for m in movie5 if random()<1.1]

In [ ]:
len(movie5_sublist)

In [ ]:
131*167

In [ ]:
sub_critics = {}
for u in user_list:
    sub_critics[u] = {}
    for m in critics[u]:
        if m in movie5_sublist:
            sub_critics[u][m] = critics[u][m]

In [ ]:
sub_critics

попробуем простейший вариант стохастического градиентного спуска

In [ ]:
seed(6660)
L = 5
Pi = {}
Bi = {}
for u in user_list:
    Bi[u] = 0
    Pi[u] = []
    for i in range(L):
        Pi[u].append(random())
Ba = {}
Qa = {}        
for m in movie5_sublist:
    Ba[m] = 0 
    Qa[m] = []
    for i in range(L):
        Qa[m].append(random())


In [ ]:
rating_list = []
for u in sub_critics:
    for m in sub_critics[u]:
        rating_list.append((u,m))

In [ ]:
len(rating_list)

In [ ]:
def evaluate_ratings(rating_list, critics, Pi, Qa, Bi, Ba, L = 5):
    summ = 0
    for (u,m) in rating_list:
        rate = Bi[u] + Ba[m]
        for i in range(L):
            rate += Pi[u][i]*Qa[m][i]
        summ += (critics[u][m] - rate)*(critics[u][m] - rate)
    return (summ/len(rating_list))**0.5
        

In [ ]:
evaluate_ratings(rating_list, sub_critics, Pi, Qa, Bi, Ba, L = 5)

In [ ]:
evaluate_ratings(rating_list, sub_critics, Pi, Qa, Bi, Ba, L = 5)

In [ ]:
gamma = 0.01

In [ ]:
p40_1 = []
p40_2 = []

In [ ]:
%%time
for jjj in range(10):
    for (u, m) in rating_list:
    #one step of gradient
        rate = Bi[u] + Ba[m]
        for i in range(L):
            rate += Pi[u][i]*Qa[m][i]
        error = rate - sub_critics[u][m]
        deltaBi = gamma*error     
        deltaBa = gamma*error     
        Bi[u] -= deltaBi
        Ba[m] -= deltaBa
        for i in range(L):
            deltaPi = (gamma*error*Qa[m][i])
            deltaQa = (gamma*error*Pi[u][i])
            Pi[u][i] -= deltaPi
            Qa[m][i] -= deltaQa
    p40_1.append(Pi["40"][1])
    p40_2.append(Pi["40"][2])
print(evaluate_ratings(rating_list, sub_critics, Pi, Qa, Bi, Ba, L = 5))

In [ ]:
evaluate_ratings(rating_list, sub_critics, Pi, Qa, Bi, Ba, L = 5)

In [ ]:
import seaborn as sns

In [ ]:
x = [i for i in range(len(p40_1))]
p40_1[:10]


In [ ]:
sns.lineplot(x = x[0:len(p40_1)], y=p40_1[0:len(p40_1)])


In [ ]:
sns.lineplot(x = x[:100], y=p40_2[:100])

In [ ]:
Pi['40']

In [ ]:
Qa

In [ ]:
movies_sorted_by_1 = sorted([m for m in Qa], key = lambda x:Qa[x][1])

In [ ]:
movies_sorted_by_1[:5]

In [ ]:
movies_sorted_by_1[-5:]

In [ ]:
movies_sorted_by_2 = sorted([m for m in Qa], key = lambda x:Qa[x][2])

In [ ]:
movies_sorted_by_2[:5]

In [ ]:
movies_sorted_by_3 = sorted([m for m in Qa], key = lambda x:Qa[x][3])

In [ ]:
movies_sorted_by_3[:5]

Чем заняться:

### Задание 1. Baseline-модель

Обучите только смещения пользователей и фильмов, без латентных факторов:

$$
\hat r_{u,i} = \mu + b_u + b_i.
$$

Сравните RMSE с полной SVD-моделью. Насколько латентные факторы улучшают результат?

Обратите внимание, что в коде нет $\mu$. Добавьте $\mu$ как неизменияемый параметр, который можно задавать. Сравните разные варианты значения $\mu$ как они влияют на сходимость? 
Правда ли, что лучшее $\mu$ это среднее по всем рейтингам?

---



### Задание 2. Влияние числа факторов

Обучите модель при $L \in \{2, 5, 10, 20, 50\}$. Постройте график зависимости итогового RMSE от $L$:

$$
L \;\mapsto\; \text{RMSE}(L).
$$

Где начинается переобучение? Как меняется поведение модели при $L \to \infty$?

---

### Задание 3. Скорость обучения

Попробуйте $\gamma \in \{0.001, 0.005, 0.01, 0.05\}$. Постройте графики сходимости:

$$
\text{RMSE}(\text{эпоха}) \quad \text{для каждого } \gamma.
$$

Что происходит при слишком большом $\gamma$? Возникает ли расходимость?

---

### Задание 4. Train/test split

Разделите оценки на тренировочные (80%) и тестовые (20%). Обучайте модель на тренировочных, считайте RMSE на тестовых:

$$
\text{RMSE}_{\text{test}} = \sqrt{\frac{1}{|\Omega_{\text{test}}|} \sum_{(u,i) \in \Omega_{\text{test}}} \bigl(r_{u,i} - \hat r_{u,i}\bigr)^2}.
$$

Как меняется переобучение с ростом $L$?

---

### Задание 5. L2-регуляризация

Добавьте в обновление параметров слагаемое $-\gamma \lambda \cdot \theta$:

$$
p_{u,j} \leftarrow p_{u,j} + \gamma\bigl(e_{u,i}\, q_{i,j} - \lambda\, p_{u,j}\bigr),
$$

$$
q_{i,j} \leftarrow q_{i,j} + \gamma\bigl(e_{u,i}\, p_{u,j} - \lambda\, q_{i,j}\bigr).
$$

Попробуйте $\lambda \in \{0.01, 0.1, 0.5\}$. Как это влияет на тестовый RMSE? Постройте зависимость $\text{RMSE}(\lambda)$.

---

### Задание 6. Early stopping

Реализуйте остановку, когда валидационный RMSE перестаёт улучшаться:

$$
\text{если } \text{RMSE}_{\text{val}}^{(t)} > \text{RMSE}_{\text{val}}^{(t-1)} \text{ в течение } k \text{ эпох} \;\Rightarrow\; \text{stop}.
$$

Сравните с фиксированным числом эпох (например, 100). Сколько эпох можно сэкономить?

---

### Задание 7. Momentum

Модифицируйте SGD, добавив momentum:

$$
\Delta_t = \beta\, \Delta_{t-1} + \gamma\, \nabla_\theta L,
$$

$$
\theta \leftarrow \theta - \Delta_t, \qquad \beta = 0.9.
$$

Ускоряет ли это сходимость? Сравните кривые RMSE от эпохи с обычным SGD.
